<a href="https://colab.research.google.com/github/indfir/indfir/blob/main/McD_Settlement_Report_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Gmail API Auto Email
# Google Colab Script - Complete Version (Fixed <NA> issue)
# ============================================================

# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client -q
print("✅ Installation complete!\n")

# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'

# Email settings
TO_EMAIL = 'indrafirdaus.youtap@gmail.com'
FROM_EMAIL = 'me'  # Gmail API akan pakai akun yang Anda authenticate

# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime
import numpy as np

print("🔄 Querying BigQuery...")

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

sql = """
WITH r AS (
  SELECT
    CASE
      WHEN BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      WHEN BIT_NAME = 'SISANYA'            THEN 'SHOPEEPAY OFF US'
      ELSE NULL
    END AS issuer,
    transaction_date,
    transaction_time,
    bank_name,
    comp_code,
    comp_name,
    store_code,
    store_name,
    mid,
    tid,
    yt_ref,
    approval_code,
    stan,
    transaction_id,
    bit_name,
    amount,
    mdr_amount,
    net_amount,
    upload_date,
    trading_date,
    pos_no,
    trx_no,
    promo_tag,
    promo_original_amount,
    payment_date,
    payment_detail,
    total_payment_amount,
    payment_trx_recon,
    voucher_code,
    issuer_trx_id
  FROM `datawarehouses.yti_settlement_mcd_report_hourly`
  WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT * FROM r
ORDER BY transaction_date DESC, transaction_time ASC
"""

df = client.query(sql).to_dataframe()

print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")

if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit

# **FIX: Replace pandas <NA> dengan None untuk Excel compatibility**
df = df.fillna('')  # atau bisa pakai df.replace({pd.NA: None})

print(f"\n📊 Data preview:")
print(df.head())

# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

print("\n📝 Generating XLSX with multiple sheets...")

wb = Workbook()
wb.remove(wb.active)

# Sheet ALL
print("  ✓ Sheet: ALL")
ws_all = wb.create_sheet('ALL')
for row in dataframe_to_rows(df, index=False, header=True):
    ws_all.append(row)
ws_all.freeze_panes = 'A2'

# Sheet per issuer
issuers = df[df['issuer'] != '']['issuer'].unique()  # skip empty issuer untuk grouping
for issuer in sorted(issuers):
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df[df['issuer'] == issuer]

    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)
    ws.freeze_panes = 'A2'

# Handle empty/NULL issuer
df_unknown = df[df['issuer'] == '']
if not df_unknown.empty:
    print(f"  ✓ Sheet: UNKNOWN ({len(df_unknown)} rows)")
    ws_unk = wb.create_sheet('UNKNOWN')
    for row in dataframe_to_rows(df_unknown, index=False, header=True):
        ws_unk.append(row)
    ws_unk.freeze_panes = 'A2'

# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)

print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")

# -------------------- SEND EMAIL VIA GMAIL API --------------------
from googleapiclient.discovery import build
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders
import base64

print("\n📧 Sending email via Gmail API...")

try:
    # Build Gmail service
    service = build('gmail', 'v1')

    # Create email message
    message = MIMEMultipart()
    message['to'] = TO_EMAIL
    message['subject'] = f'[BQ Export] Settlement Report {ts}'

    # Email body
    body_text = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary:
- Total Rows: {len(df):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: 1 hari terakhir
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}
"""

    message.attach(MIMEText(body_text, 'plain'))

    # Attach XLSX file
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    message.attach(part)

    # Encode and send
    raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode('utf-8')
    send_message = service.users().messages().send(
        userId='me',
        body={'raw': raw_message}
    ).execute()

    print(f"✅ Email sent successfully!")
    print(f"   Message ID: {send_message['id']}")
    print(f"   Sent to: {TO_EMAIL}")

except Exception as e:
    print(f"❌ Email failed: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Enable Gmail API: https://console.cloud.google.com/apis/library/gmail.googleapis.com")
    print("2. Select project: youtap-indonesia-bi")
    print("3. Re-run this script from the beginning")
    raise

print("\n🎉 Process complete!")
print(f"📁 File: {filename}")
print(f"📧 Check inbox: {TO_EMAIL}")


📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 18,481 rows, 30 columns


TypeError: Invalid value '' for dtype Int64

In [2]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Gmail API Auto Email
# Google Colab Script - Complete Version (Fixed Int64 issue)
# ============================================================

# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client -q
print("✅ Installation complete!\n")

# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'

# Email settings
TO_EMAIL = 'indrafirdaus.youtap@gmail.com'
FROM_EMAIL = 'me'  # Gmail API akan pakai akun yang Anda authenticate

# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime
import numpy as np

print("🔄 Querying BigQuery...")

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

sql = """
WITH r AS (
  SELECT
    CASE
      WHEN BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      WHEN BIT_NAME = 'SISANYA'            THEN 'SHOPEEPAY OFF US'
      ELSE NULL
    END AS issuer,
    transaction_date,
    transaction_time,
    bank_name,
    comp_code,
    comp_name,
    store_code,
    store_name,
    mid,
    tid,
    yt_ref,
    approval_code,
    stan,
    transaction_id,
    bit_name,
    amount,
    mdr_amount,
    net_amount,
    upload_date,
    trading_date,
    pos_no,
    trx_no,
    promo_tag,
    promo_original_amount,
    payment_date,
    payment_detail,
    total_payment_amount,
    payment_trx_recon,
    voucher_code,
    issuer_trx_id
  FROM `datawarehouses.yti_settlement_mcd_report_hourly`
  WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT * FROM r
ORDER BY transaction_date DESC, transaction_time ASC
"""

df = client.query(sql).to_dataframe()

print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")

if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit

# **FIX: Convert semua kolom ke object type (string) supaya Excel-friendly**
print("🔧 Converting data types for Excel compatibility...")
for col in df.columns:
    # Convert semua ke string, lalu replace 'nan', 'None', '<NA>' jadi empty string
    df[col] = df[col].astype(str).replace(['nan', 'None', '<NA>', 'NaT'], '')

print(f"✅ Data conversion complete!")
print(f"\n📊 Data preview:")
print(df.head())

# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

print("\n📝 Generating XLSX with multiple sheets...")

wb = Workbook()
wb.remove(wb.active)

# Sheet ALL
print("  ✓ Sheet: ALL")
ws_all = wb.create_sheet('ALL')
for row in dataframe_to_rows(df, index=False, header=True):
    ws_all.append(row)
ws_all.freeze_panes = 'A2'

# Sheet per issuer
issuers = df[df['issuer'] != '']['issuer'].unique()
for issuer in sorted(issuers):
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '').replace(':', '-')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df[df['issuer'] == issuer]

    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)
    ws.freeze_panes = 'A2'

# Handle empty/NULL issuer
df_unknown = df[df['issuer'] == '']
if not df_unknown.empty:
    print(f"  ✓ Sheet: UNKNOWN ({len(df_unknown)} rows)")
    ws_unk = wb.create_sheet('UNKNOWN')
    for row in dataframe_to_rows(df_unknown, index=False, header=True):
        ws_unk.append(row)
    ws_unk.freeze_panes = 'A2'

# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)

print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")

# -------------------- SEND EMAIL VIA GMAIL API --------------------
from googleapiclient.discovery import build
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders
import base64

print("\n📧 Sending email via Gmail API...")

try:
    # Build Gmail service
    service = build('gmail', 'v1')

    # Create email message
    message = MIMEMultipart()
    message['to'] = TO_EMAIL
    message['subject'] = f'[BQ Export] Settlement Report {ts}'

    # Email body
    body_text = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary:
- Total Rows: {len(df):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: 1 hari terakhir
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}
"""

    message.attach(MIMEText(body_text, 'plain'))

    # Attach XLSX file
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    message.attach(part)

    # Encode and send
    raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode('utf-8')
    send_message = service.users().messages().send(
        userId='me',
        body={'raw': raw_message}
    ).execute()

    print(f"✅ Email sent successfully!")
    print(f"   Message ID: {send_message['id']}")
    print(f"   Sent to: {TO_EMAIL}")

except Exception as e:
    print(f"❌ Email failed: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Enable Gmail API: https://console.cloud.google.com/apis/library/gmail.googleapis.com")
    print("2. Select project: youtap-indonesia-bi")
    print("3. Click 'ENABLE'")
    print("4. Re-run this script")
    import traceback
    traceback.print_exc()

print("\n🎉 Process complete!")
print(f"📁 File: {filename}")
print(f"📧 Check inbox: {TO_EMAIL}")


📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 18,481 rows, 30 columns
🔧 Converting data types for Excel compatibility...
✅ Data conversion complete!

📊 Data preview:
  issuer transaction_date transaction_time bank_name comp_code comp_name  \
0  LIVIN       2026-02-06         00:00:17    Youtap        03       CSO   
1  LIVIN       2026-02-06         00:00:30    Youtap        03       CSO   
2    BNI       2026-02-06         00:00:33    Youtap        03       CSO   
3  LIVIN       2026-02-06         00:00:34    Youtap        03       CSO   
4  LIVIN       2026-02-06         00:00:35    Youtap        03       CSO   

  store_code                      store_name           mid           tid  ...  \
0   19700430                McD Manahan Solo  621970043003  ID00430CSO32  ...   
1   19700231               McD Jombor Sleman  621970023103  ID00231CSO32  ...   
2   19700183  McD M

❌ Email failed: <HttpError 403 when requesting https://gmail.googleapis.com/gmail/v1/users/me/messages/send?alt=json returned "Request had insufficient authentication scopes.". Details: "[{'message': 'Insufficient Permission', 'domain': 'global', 'reason': 'insufficientPermissions'}]">

Troubleshooting:
1. Enable Gmail API: https://console.cloud.google.com/apis/library/gmail.googleapis.com
2. Select project: youtap-indonesia-bi
3. Click 'ENABLE'
4. Re-run this script

🎉 Process complete!
📁 File: mcd_settlement_20260206_035241.xlsx
📧 Check inbox: indrafirdaus.youtap@gmail.com


Traceback (most recent call last):
  File "/tmp/ipython-input-3435118632.py", line 196, in <cell line: 0>
    ).execute()
      ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/googleapiclient/_helpers.py", line 130, in positional_wrapper
    return wrapped(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/googleapiclient/http.py", line 938, in execute
    raise HttpError(resp, content, uri=self.uri)
googleapiclient.errors.HttpError: <HttpError 403 when requesting https://gmail.googleapis.com/gmail/v1/users/me/messages/send?alt=json returned "Request had insufficient authentication scopes.". Details: "[{'message': 'Insufficient Permission', 'domain': 'global', 'reason': 'insufficientPermissions'}]">


In [3]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Email via SMTP
# Google Colab Script - SIMPLEST VERSION (No API Setup!)
# ============================================================

# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow -q
print("✅ Installation complete!\n")

# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'

# Email settings
TO_EMAIL = 'indra.firdaus@youtap.com'  # Email kantor Anda
FROM_EMAIL = 'indrafirdaus.youtap@gmail.com'  # Gmail Anda untuk ngirim

# Gmail App Password - WAJIB ISI (cara dapat: lihat di bawah)
GMAIL_APP_PASSWORD = 'PASTE_APP_PASSWORD_HERE'  # <--- ISI INI! (16 karakter tanpa spasi)

# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

print("🔄 Querying BigQuery...")

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

sql = """
WITH r AS (
  SELECT
    CASE
      WHEN BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      WHEN BIT_NAME = 'SISANYA'            THEN 'SHOPEEPAY OFF US'
      ELSE NULL
    END AS issuer,
    transaction_date,
    transaction_time,
    bank_name,
    comp_code,
    comp_name,
    store_code,
    store_name,
    mid,
    tid,
    yt_ref,
    approval_code,
    stan,
    transaction_id,
    bit_name,
    amount,
    mdr_amount,
    net_amount,
    upload_date,
    trading_date,
    pos_no,
    trx_no,
    promo_tag,
    promo_original_amount,
    payment_date,
    payment_detail,
    total_payment_amount,
    payment_trx_recon,
    voucher_code,
    issuer_trx_id
  FROM `datawarehouses.yti_settlement_mcd_report_hourly`
  WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT * FROM r
ORDER BY transaction_date DESC, transaction_time ASC
"""

df = client.query(sql).to_dataframe()

print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")

if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit

# Convert semua kolom ke string untuk Excel compatibility
print("🔧 Converting data types for Excel compatibility...")
for col in df.columns:
    df[col] = df[col].astype(str).replace(['nan', 'None', '<NA>', 'NaT'], '')

print(f"✅ Data conversion complete!")

# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

print("\n📝 Generating XLSX with multiple sheets...")

wb = Workbook()
wb.remove(wb.active)

# Sheet ALL
print("  ✓ Sheet: ALL")
ws_all = wb.create_sheet('ALL')
for row in dataframe_to_rows(df, index=False, header=True):
    ws_all.append(row)
ws_all.freeze_panes = 'A2'

# Sheet per issuer
issuers = df[df['issuer'] != '']['issuer'].unique()
for issuer in sorted(issuers):
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '').replace(':', '-')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df[df['issuer'] == issuer]

    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)
    ws.freeze_panes = 'A2'

# Handle empty/NULL issuer
df_unknown = df[df['issuer'] == '']
if not df_unknown.empty:
    print(f"  ✓ Sheet: UNKNOWN ({len(df_unknown)} rows)")
    ws_unk = wb.create_sheet('UNKNOWN')
    for row in dataframe_to_rows(df_unknown, index=False, header=True):
        ws_unk.append(row)
    ws_unk.freeze_panes = 'A2'

# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)

print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")

# -------------------- SEND EMAIL VIA SMTP --------------------
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

print("\n📧 Sending email via SMTP...")

# Validation
if GMAIL_APP_PASSWORD == 'PASTE_APP_PASSWORD_HERE':
    print("❌ ERROR: Please set GMAIL_APP_PASSWORD!")
    print("\n📝 Cara dapat App Password:")
    print("1. Buka: https://myaccount.google.com/apppasswords")
    print("2. Login dengan Gmail Anda: indrafirdaus.youtap@gmail.com")
    print("3. Ketik nama app: 'Colab Settlement Export'")
    print("4. Click 'Create'")
    print("5. Copy 16-digit password yang muncul")
    print("6. Paste di variable GMAIL_APP_PASSWORD (tanpa spasi)")
    print("\n📥 Downloading file instead...")
    from google.colab import files
    files.download(filename)
    raise SystemExit

try:
    # Create email
    msg = MIMEMultipart()
    msg['From'] = FROM_EMAIL
    msg['To'] = TO_EMAIL
    msg['Subject'] = f'[BQ Export] Settlement Report {ts}'

    # Email body
    body = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary:
- Total Rows: {len(df):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: 1 hari terakhir
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}

---
Auto-generated via Google Colab
"""

    msg.attach(MIMEText(body, 'plain'))

    # Attach XLSX
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    msg.attach(part)

    # Send via Gmail SMTP
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(FROM_EMAIL, GMAIL_APP_PASSWORD)
    server.send_message(msg)
    server.quit()

    print(f"✅ Email sent successfully!")
    print(f"   From: {FROM_EMAIL}")
    print(f"   To: {TO_EMAIL}")
    print(f"   Check inbox in 10-30 seconds!")

except Exception as e:
    print(f"❌ Email failed: {str(e)}")
    print(f"\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)

print("\n🎉 Process complete!")
print(f"📁 File: {filename}")


📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 18,481 rows, 30 columns
🔧 Converting data types for Excel compatibility...
✅ Data conversion complete!

📝 Generating XLSX with multiple sheets...
  ✓ Sheet: ALL
  ✓ Sheet: BNI
  ✓ Sheet: INDODANA
  ✓ Sheet: KREDIVO
  ✓ Sheet: LIVIN
  ✓ Sheet: SHOPEEPAY
  ✓ Sheet: UNKNOWN (5141 rows)

✅ XLSX created: mcd_settlement_20260206_035724.xlsx
   Total sheets: 7 → ALL, BNI, INDODANA, KREDIVO, LIVIN, SHOPEEPAY, UNKNOWN

📧 Sending email via SMTP...
❌ ERROR: Please set GMAIL_APP_PASSWORD!

📝 Cara dapat App Password:
1. Buka: https://myaccount.google.com/apppasswords
2. Login dengan Gmail Anda: indrafirdaus.youtap@gmail.com
3. Ketik nama app: 'Colab Settlement Export'
4. Click 'Create'
5. Copy 16-digit password yang muncul
6. Paste di variable GMAIL_APP_PASSWORD (tanpa spasi)

📥 Downloading file instead...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

SystemExit: 

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Email via SMTP
# Google Colab Script - SIMPLEST VERSION (No API Setup!)
# ============================================================

# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow -q
print("✅ Installation complete!\n")

# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'

# Email settings
TO_EMAIL = 'indra.firdaus@youtap.com'  # Email kantor Anda
FROM_EMAIL = 'indrafirdaus.youtap@gmail.com'  # Gmail Anda untuk ngirim

# Gmail App Password - WAJIB ISI (cara dapat: lihat di bawah)
GMAIL_APP_PASSWORD = 'PASTE_APP_PASSWORD_HERE'  # <--- ISI INI! (16 karakter tanpa spasi)

# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

print("🔄 Querying BigQuery...")

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

sql = """
WITH r AS (
  SELECT
    CASE
      WHEN BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      ELSE 'SHOPEEPAY OFF US'
    END AS issuer,
    transaction_date,
    transaction_time,
    bank_name,
    comp_code,
    comp_name,
    store_code,
    store_name,
    mid,
    tid,
    yt_ref,
    approval_code,
    stan,
    transaction_id,
    bit_name,
    amount,
    mdr_amount,
    net_amount,
    upload_date,
    trading_date,
    pos_no,
    trx_no,
    promo_tag,
    promo_original_amount,
    payment_date,
    payment_detail,
    total_payment_amount,
    payment_trx_recon,
    voucher_code,
    issuer_trx_id
  FROM `datawarehouses.yti_settlement_mcd_report_hourly`
  WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT * FROM r
ORDER BY transaction_date DESC, transaction_time ASC
"""

df = client.query(sql).to_dataframe()

print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")

if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit

# Convert semua kolom ke string untuk Excel compatibility
print("🔧 Converting data types for Excel compatibility...")
for col in df.columns:
    df[col] = df[col].astype(str).replace(['nan', 'None', '<NA>', 'NaT'], '')

print(f"✅ Data conversion complete!")

# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

print("\n📝 Generating XLSX with multiple sheets...")

wb = Workbook()
wb.remove(wb.active)

# Sheet ALL
print("  ✓ Sheet: ALL")
ws_all = wb.create_sheet('ALL')
for row in dataframe_to_rows(df, index=False, header=True):
    ws_all.append(row)
ws_all.freeze_panes = 'A2'

# Sheet per issuer
issuers = df[df['issuer'] != '']['issuer'].unique()
for issuer in sorted(issuers):
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '').replace(':', '-')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df[df['issuer'] == issuer]

    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)
    ws.freeze_panes = 'A2'

# Handle empty/NULL issuer
df_unknown = df[df['issuer'] == '']
if not df_unknown.empty:
    print(f"  ✓ Sheet: UNKNOWN ({len(df_unknown)} rows)")
    ws_unk = wb.create_sheet('UNKNOWN')
    for row in dataframe_to_rows(df_unknown, index=False, header=True):
        ws_unk.append(row)
    ws_unk.freeze_panes = 'A2'

# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)

print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")

# -------------------- SEND EMAIL VIA SMTP --------------------
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

print("\n📧 Sending email via SMTP...")

# Validation
if GMAIL_APP_PASSWORD == 'kjfpelblsfhyadoa':
    print("❌ ERROR: Please set GMAIL_APP_PASSWORD!")
    print("\n📝 Cara dapat App Password:")
    print("1. Buka: https://myaccount.google.com/apppasswords")
    print("2. Login dengan Gmail Anda: indrafirdaus.youtap@gmail.com")
    print("3. Ketik nama app: 'Colab Settlement Export'")
    print("4. Click 'Create'")
    print("5. Copy 16-digit password yang muncul")
    print("6. Paste di variable GMAIL_APP_PASSWORD (tanpa spasi)")
    print("\n📥 Downloading file instead...")
    from google.colab import files
    files.download(filename)
    raise SystemExit

try:
    # Create email
    msg = MIMEMultipart()
    msg['From'] = FROM_EMAIL
    msg['To'] = TO_EMAIL
    msg['Subject'] = f'[BQ Export] Settlement Report {ts}'

    # Email body
    body = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary:
- Total Rows: {len(df):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: 1 hari terakhir
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}

---
Auto-generated via Google Colab
"""

    msg.attach(MIMEText(body, 'plain'))

    # Attach XLSX
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    msg.attach(part)

    # Send via Gmail SMTP
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(FROM_EMAIL, GMAIL_APP_PASSWORD)
    server.send_message(msg)
    server.quit()

    print(f"✅ Email sent successfully!")
    print(f"   From: {FROM_EMAIL}")
    print(f"   To: {TO_EMAIL}")
    print(f"   Check inbox in 10-30 seconds!")

except Exception as e:
    print(f"❌ Email failed: {str(e)}")
    print(f"\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)

print("\n🎉 Process complete!")
print(f"📁 File: {filename}")


📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 18,481 rows, 30 columns
🔧 Converting data types for Excel compatibility...
✅ Data conversion complete!

📝 Generating XLSX with multiple sheets...
  ✓ Sheet: ALL
  ✓ Sheet: BNI
  ✓ Sheet: INDODANA
  ✓ Sheet: KREDIVO
  ✓ Sheet: LIVIN
  ✓ Sheet: SHOPEEPAY
  ✓ Sheet: SHOPEEPAY OFF US

✅ XLSX created: mcd_settlement_20260206_040028.xlsx
   Total sheets: 7 → ALL, BNI, INDODANA, KREDIVO, LIVIN, SHOPEEPAY, SHOPEEPAY OFF US

📧 Sending email via SMTP...
❌ Email failed: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials 6a1803df08f44-8953bf58af0sm10307986d6.16 - gsmtp')

📥 Downloading file as fallback...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Process complete!
📁 File: mcd_settlement_20260206_040028.xlsx


In [5]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Email via SMTP
# Google Colab Script - SIMPLEST VERSION (No API Setup!)
# ============================================================

# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow -q
print("✅ Installation complete!\n")

# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'

# Email settings
TO_EMAIL = 'indra.firdaus@youtap.com'  # Email kantor Anda
FROM_EMAIL = 'indrafirdaus.youtap@gmail.com'  # Gmail Anda untuk ngirim

# Gmail App Password - WAJIB ISI (cara dapat: lihat di bawah)
GMAIL_APP_PASSWORD = 'PASTE_APP_PASSWORD_HERE'  # <--- ISI INI! (16 karakter tanpa spasi)

# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

print("🔄 Querying BigQuery...")

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

sql = """
WITH r AS (
  SELECT
    CASE
      WHEN BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      ELSE 'SHOPEEPAY OFF US'
    END AS issuer,
    transaction_date,
    transaction_time,
    bank_name,
    comp_code,
    comp_name,
    store_code,
    store_name,
    mid,
    tid,
    yt_ref,
    approval_code,
    stan,
    transaction_id,
    bit_name,
    amount,
    mdr_amount,
    net_amount,
    upload_date,
    trading_date,
    pos_no,
    trx_no,
    promo_tag,
    promo_original_amount,
    payment_date,
    payment_detail,
    total_payment_amount,
    payment_trx_recon,
    voucher_code,
    issuer_trx_id
  FROM `datawarehouses.yti_settlement_mcd_report_hourly`
  WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT * FROM r
ORDER BY transaction_date DESC, transaction_time ASC
"""

df = client.query(sql).to_dataframe()

print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")

if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit

# Convert semua kolom ke string untuk Excel compatibility
print("🔧 Converting data types for Excel compatibility...")
for col in df.columns:
    df[col] = df[col].astype(str).replace(['nan', 'None', '<NA>', 'NaT'], '')

print(f"✅ Data conversion complete!")

# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

print("\n📝 Generating XLSX with multiple sheets...")

wb = Workbook()
wb.remove(wb.active)

# Sheet ALL
print("  ✓ Sheet: ALL")
ws_all = wb.create_sheet('ALL')
for row in dataframe_to_rows(df, index=False, header=True):
    ws_all.append(row)
ws_all.freeze_panes = 'A2'

# Sheet per issuer
issuers = df[df['issuer'] != '']['issuer'].unique()
for issuer in sorted(issuers):
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '').replace(':', '-')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df[df['issuer'] == issuer]

    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)
    ws.freeze_panes = 'A2'

# Handle empty/NULL issuer
df_unknown = df[df['issuer'] == '']
if not df_unknown.empty:
    print(f"  ✓ Sheet: UNKNOWN ({len(df_unknown)} rows)")
    ws_unk = wb.create_sheet('UNKNOWN')
    for row in dataframe_to_rows(df_unknown, index=False, header=True):
        ws_unk.append(row)
    ws_unk.freeze_panes = 'A2'

# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)

print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")

# -------------------- SEND EMAIL VIA SMTP --------------------
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

print("\n📧 Sending email via SMTP...")

# Validation
if GMAIL_APP_PASSWORD == 'kjfp elbl sfhy adoa':
    print("❌ ERROR: Please set GMAIL_APP_PASSWORD!")
    print("\n📝 Cara dapat App Password:")
    print("1. Buka: https://myaccount.google.com/apppasswords")
    print("2. Login dengan Gmail Anda: indrafirdaus.youtap@gmail.com")
    print("3. Ketik nama app: 'Colab Settlement Export'")
    print("4. Click 'Create'")
    print("5. Copy 16-digit password yang muncul")
    print("6. Paste di variable GMAIL_APP_PASSWORD (tanpa spasi)")
    print("\n📥 Downloading file instead...")
    from google.colab import files
    files.download(filename)
    raise SystemExit

try:
    # Create email
    msg = MIMEMultipart()
    msg['From'] = FROM_EMAIL
    msg['To'] = TO_EMAIL
    msg['Subject'] = f'[BQ Export] Settlement Report {ts}'

    # Email body
    body = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary:
- Total Rows: {len(df):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: 1 hari terakhir
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}

---
Auto-generated via Google Colab
"""

    msg.attach(MIMEText(body, 'plain'))

    # Attach XLSX
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    msg.attach(part)

    # Send via Gmail SMTP
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(FROM_EMAIL, GMAIL_APP_PASSWORD)
    server.send_message(msg)
    server.quit()

    print(f"✅ Email sent successfully!")
    print(f"   From: {FROM_EMAIL}")
    print(f"   To: {TO_EMAIL}")
    print(f"   Check inbox in 10-30 seconds!")

except Exception as e:
    print(f"❌ Email failed: {str(e)}")
    print(f"\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)

print("\n🎉 Process complete!")
print(f"📁 File: {filename}")

📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 18,481 rows, 30 columns
🔧 Converting data types for Excel compatibility...
✅ Data conversion complete!

📝 Generating XLSX with multiple sheets...
  ✓ Sheet: ALL
  ✓ Sheet: BNI
  ✓ Sheet: INDODANA
  ✓ Sheet: KREDIVO
  ✓ Sheet: LIVIN
  ✓ Sheet: SHOPEEPAY
  ✓ Sheet: SHOPEEPAY OFF US

✅ XLSX created: mcd_settlement_20260206_040231.xlsx
   Total sheets: 7 → ALL, BNI, INDODANA, KREDIVO, LIVIN, SHOPEEPAY, SHOPEEPAY OFF US

📧 Sending email via SMTP...
❌ Email failed: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials 6a1803df08f44-8953bf71f53sm11013576d6.25 - gsmtp')

📥 Downloading file as fallback...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Process complete!
📁 File: mcd_settlement_20260206_040231.xlsx


In [6]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Email via SMTP
# Google Colab Script - SIMPLEST VERSION (No API Setup!)
# ============================================================

# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow -q
print("✅ Installation complete!\n")

# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'

# Email settings
TO_EMAIL = 'indra.firdaus@youtap.com'  # Email kantor Anda
FROM_EMAIL = 'indrafirdaus.youtap@gmail.com'  # Gmail Anda untuk ngirim

# Gmail App Password - WAJIB ISI (cara dapat: lihat di bawah)
GMAIL_APP_PASSWORD = 'PASTE_APP_PASSWORD_HERE'  # <--- ISI INI! (16 karakter tanpa spasi)

# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

print("🔄 Querying BigQuery...")

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

sql = """
WITH r AS (
  SELECT
    CASE
      WHEN BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      ELSE 'SHOPEEPAY OFF US'
    END AS issuer,
    transaction_date,
    transaction_time,
    bank_name,
    comp_code,
    comp_name,
    store_code,
    store_name,
    mid,
    tid,
    yt_ref,
    approval_code,
    stan,
    transaction_id,
    bit_name,
    amount,
    mdr_amount,
    net_amount,
    upload_date,
    trading_date,
    pos_no,
    trx_no,
    promo_tag,
    promo_original_amount,
    payment_date,
    payment_detail,
    total_payment_amount,
    payment_trx_recon,
    voucher_code,
    issuer_trx_id
  FROM `datawarehouses.yti_settlement_mcd_report_hourly`
  WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT * FROM r
ORDER BY transaction_date DESC, transaction_time ASC
"""

df = client.query(sql).to_dataframe()

print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")

if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit

# Convert semua kolom ke string untuk Excel compatibility
print("🔧 Converting data types for Excel compatibility...")
for col in df.columns:
    df[col] = df[col].astype(str).replace(['nan', 'None', '<NA>', 'NaT'], '')

print(f"✅ Data conversion complete!")

# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

print("\n📝 Generating XLSX with multiple sheets...")

wb = Workbook()
wb.remove(wb.active)

# Sheet ALL
print("  ✓ Sheet: ALL")
ws_all = wb.create_sheet('ALL')
for row in dataframe_to_rows(df, index=False, header=True):
    ws_all.append(row)
ws_all.freeze_panes = 'A2'

# Sheet per issuer
issuers = df[df['issuer'] != '']['issuer'].unique()
for issuer in sorted(issuers):
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '').replace(':', '-')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df[df['issuer'] == issuer]

    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)
    ws.freeze_panes = 'A2'

# Handle empty/NULL issuer
df_unknown = df[df['issuer'] == '']
if not df_unknown.empty:
    print(f"  ✓ Sheet: UNKNOWN ({len(df_unknown)} rows)")
    ws_unk = wb.create_sheet('UNKNOWN')
    for row in dataframe_to_rows(df_unknown, index=False, header=True):
        ws_unk.append(row)
    ws_unk.freeze_panes = 'A2'

# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)

print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")

# -------------------- SEND EMAIL VIA SMTP --------------------
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

print("\n📧 Sending email via SMTP...")

# Validation
if GMAIL_APP_PASSWORD == 'jotvmycoowpimgni':
    print("❌ ERROR: Please set GMAIL_APP_PASSWORD!")
    print("\n📝 Cara dapat App Password:")
    print("1. Buka: https://myaccount.google.com/apppasswords")
    print("2. Login dengan Gmail Anda: indrafirdaus.youtap@gmail.com")
    print("3. Ketik nama app: 'Colab Settlement Export'")
    print("4. Click 'Create'")
    print("5. Copy 16-digit password yang muncul")
    print("6. Paste di variable GMAIL_APP_PASSWORD (tanpa spasi)")
    print("\n📥 Downloading file instead...")
    from google.colab import files
    files.download(filename)
    raise SystemExit

try:
    # Create email
    msg = MIMEMultipart()
    msg['From'] = FROM_EMAIL
    msg['To'] = TO_EMAIL
    msg['Subject'] = f'[BQ Export] Settlement Report {ts}'

    # Email body
    body = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary:
- Total Rows: {len(df):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: 1 hari terakhir
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}

---
Auto-generated via Google Colab
"""

    msg.attach(MIMEText(body, 'plain'))

    # Attach XLSX
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    msg.attach(part)

    # Send via Gmail SMTP
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(FROM_EMAIL, GMAIL_APP_PASSWORD)
    server.send_message(msg)
    server.quit()

    print(f"✅ Email sent successfully!")
    print(f"   From: {FROM_EMAIL}")
    print(f"   To: {TO_EMAIL}")
    print(f"   Check inbox in 10-30 seconds!")

except Exception as e:
    print(f"❌ Email failed: {str(e)}")
    print(f"\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)

print("\n🎉 Process complete!")
print(f"📁 File: {filename}")

📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 18,481 rows, 30 columns
🔧 Converting data types for Excel compatibility...
✅ Data conversion complete!

📝 Generating XLSX with multiple sheets...
  ✓ Sheet: ALL
  ✓ Sheet: BNI
  ✓ Sheet: INDODANA
  ✓ Sheet: KREDIVO
  ✓ Sheet: LIVIN
  ✓ Sheet: SHOPEEPAY
  ✓ Sheet: SHOPEEPAY OFF US

✅ XLSX created: mcd_settlement_20260206_040628.xlsx
   Total sheets: 7 → ALL, BNI, INDODANA, KREDIVO, LIVIN, SHOPEEPAY, SHOPEEPAY OFF US

📧 Sending email via SMTP...
❌ Email failed: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials af79cd13be357-8caf77f6c16sm76518285a.10 - gsmtp')

📥 Downloading file as fallback...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Process complete!
📁 File: mcd_settlement_20260206_040628.xlsx


In [7]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Email via SMTP
# Google Colab Script - SIMPLEST VERSION (No API Setup!)
# ============================================================


# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow -q
print("✅ Installation complete!\n")


# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'


# Email settings
TO_EMAIL = 'indra.firdaus@youtap.com'  # Email kantor Anda
FROM_EMAIL = 'indrafirdaus.youtap@gmail.com'  # Gmail Anda untuk ngirim


# Gmail App Password - WAJIB ISI (cara dapat: lihat di bawah)
# 1. Pastikan 2FA aktif di akun Gmail Anda
# 2. Buka: https://myaccount.google.com/apppasswords
# 3. Login dengan Gmail: indrafirdaus.youtap@gmail.com
# 4. Pilih App: "Mail" atau "Other" → ketik "Colab Settlement Export"
# 5. Pilih Device: "Other" → ketik "Google Colab"
# 6. Click "Generate"
# 7. Copy 16-digit password (tanpa spasi!)
# 8. Paste di bawah ini:
GMAIL_APP_PASSWORD = 'jotvmycoowpimgni'  # <--- ISI INI! (16 karakter tanpa spasi)


# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")


# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime


print("🔄 Querying BigQuery...")


client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)


sql = """
WITH r AS (
  SELECT
    CASE
      WHEN BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      ELSE 'SHOPEEPAY OFF US'
    END AS issuer,
    transaction_date,
    transaction_time,
    bank_name,
    comp_code,
    comp_name,
    store_code,
    store_name,
    mid,
    tid,
    yt_ref,
    approval_code,
    stan,
    transaction_id,
    bit_name,
    amount,
    mdr_amount,
    net_amount,
    upload_date,
    trading_date,
    pos_no,
    trx_no,
    promo_tag,
    promo_original_amount,
    payment_date,
    payment_detail,
    total_payment_amount,
    payment_trx_recon,
    voucher_code,
    issuer_trx_id
  FROM `datawarehouses.yti_settlement_mcd_report_hourly`
  WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT * FROM r
ORDER BY transaction_date DESC, transaction_time ASC
"""


df = client.query(sql).to_dataframe()


print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")


if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit


# Convert semua kolom ke string untuk Excel compatibility
print("🔧 Converting data types for Excel compatibility...")
for col in df.columns:
    df[col] = df[col].astype(str).replace(['nan', 'None', '<NA>', 'NaT'], '')


print(f"✅ Data conversion complete!")


# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows


print("\n📝 Generating XLSX with multiple sheets...")


wb = Workbook()
wb.remove(wb.active)


# Sheet ALL
print("  ✓ Sheet: ALL")
ws_all = wb.create_sheet('ALL')
for row in dataframe_to_rows(df, index=False, header=True):
    ws_all.append(row)
ws_all.freeze_panes = 'A2'


# Sheet per issuer
issuers = df[df['issuer'] != '']['issuer'].unique()
for issuer in sorted(issuers):
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '').replace(':', '-')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df[df['issuer'] == issuer]

    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)
    ws.freeze_panes = 'A2'


# Handle empty/NULL issuer
df_unknown = df[df['issuer'] == '']
if not df_unknown.empty:
    print(f"  ✓ Sheet: UNKNOWN ({len(df_unknown)} rows)")
    ws_unk = wb.create_sheet('UNKNOWN')
    for row in dataframe_to_rows(df_unknown, index=False, header=True):
        ws_unk.append(row)
    ws_unk.freeze_panes = 'A2'


# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)


print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")


# -------------------- SEND EMAIL VIA SMTP --------------------
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders


print("\n📧 Sending email via SMTP...")


# Validation
if GMAIL_APP_PASSWORD == 'jotvmycoowpimgni':
    print("❌ ERROR: Please set GMAIL_APP_PASSWORD!")
    print("\n📝 Cara dapat App Password:")
    print("1. Pastikan 2FA aktif di Gmail Anda")
    print("2. Buka: https://myaccount.google.com/apppasswords")
    print("3. Login dengan Gmail: indrafirdaus.youtap@gmail.com")
    print("4. Pilih App: 'Mail' atau 'Other (Custom name)'")
    print("5. Ketik nama: 'Colab Settlement Export'")
    print("6. Click 'Generate'")
    print("7. Copy 16-digit password (contoh: abcd efgh ijkl mnop)")
    print("8. Paste di variable GMAIL_APP_PASSWORD TANPA SPASI (abcdefghijklmnop)")
    print("\n📥 Downloading file instead...")
    from google.colab import files
    files.download(filename)
    raise SystemExit


try:
    # Create email
    msg = MIMEMultipart()
    msg['From'] = FROM_EMAIL
    msg['To'] = TO_EMAIL
    msg['Subject'] = f'[BQ Export] MCD Settlement Report {ts}'

    # Email body
    body = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary:
- Total Rows: {len(df):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: 1 hari terakhir
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}

---
Auto-generated via Google Colab
"""

    msg.attach(MIMEText(body, 'plain'))

    # Attach XLSX
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    msg.attach(part)

    # Send via Gmail SMTP
    print("🔄 Connecting to Gmail SMTP server...")
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()

    print("🔑 Authenticating with App Password...")
    server.login(FROM_EMAIL, GMAIL_APP_PASSWORD)

    print("📤 Sending email...")
    server.send_message(msg)
    server.quit()

    print(f"\n✅ Email sent successfully!")
    print(f"   From: {FROM_EMAIL}")
    print(f"   To: {TO_EMAIL}")
    print(f"   Subject: [BQ Export] MCD Settlement Report {ts}")
    print(f"   Attachment: {filename}")
    print(f"\n📬 Check your inbox at {TO_EMAIL} in 10-30 seconds!")

except smtplib.SMTPAuthenticationError as e:
    print(f"\n❌ Authentication failed: {str(e)}")
    print("\n🔍 Troubleshooting:")
    print("   1. Pastikan 2FA aktif di akun Gmail Anda")
    print("   2. Generate App Password baru di: https://myaccount.google.com/apppasswords")
    print("   3. Pastikan App Password 16 karakter TANPA SPASI")
    print("   4. Jangan gunakan password Gmail biasa!")
    print("\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)

except Exception as e:
    print(f"\n❌ Email failed: {str(e)}")
    print(f"\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)


print("\n🎉 Process complete!")
print(f"📁 File: {filename}")


📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 18,481 rows, 30 columns
🔧 Converting data types for Excel compatibility...
✅ Data conversion complete!

📝 Generating XLSX with multiple sheets...
  ✓ Sheet: ALL
  ✓ Sheet: BNI
  ✓ Sheet: INDODANA
  ✓ Sheet: KREDIVO
  ✓ Sheet: LIVIN
  ✓ Sheet: SHOPEEPAY
  ✓ Sheet: SHOPEEPAY OFF US

✅ XLSX created: mcd_settlement_20260206_041103.xlsx
   Total sheets: 7 → ALL, BNI, INDODANA, KREDIVO, LIVIN, SHOPEEPAY, SHOPEEPAY OFF US

📧 Sending email via SMTP...
❌ ERROR: Please set GMAIL_APP_PASSWORD!

📝 Cara dapat App Password:
1. Pastikan 2FA aktif di Gmail Anda
2. Buka: https://myaccount.google.com/apppasswords
3. Login dengan Gmail: indrafirdaus.youtap@gmail.com
4. Pilih App: 'Mail' atau 'Other (Custom name)'
5. Ketik nama: 'Colab Settlement Export'
6. Click 'Generate'
7. Copy 16-digit password (contoh: abcd efgh ijkl mnop)
8. Paste di variab

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

SystemExit: 

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [12]:
# ============================================================
# BigQuery → XLSX Multi-Sheet → Email via SMTP
# Google Colab Script - WITH MATCH/NOT MATCH LOGIC
# ============================================================

# -------------------- INSTALL DEPENDENCIES --------------------
print("📦 Installing dependencies...")
!pip install google-cloud-bigquery openpyxl pandas db-dtypes pyarrow -q
print("✅ Installation complete!\n")

# -------------------- CONFIGURATION --------------------
PROJECT_ID = 'youtap-indonesia-bi'
BQ_LOCATION = 'asia-southeast2'

# Email settings - Multiple recipients
TO_EMAILS = [
    'indra.firdaus@youtap.com',
    'lorenza.salindri@youtap.com'
]
FROM_EMAIL = 'youtap.bi@gmail.com'
GMAIL_APP_PASSWORD = 'vdfhjksdkmfstbvo'

# -------------------- AUTHENTICATION --------------------
from google.colab import auth
print("🔐 Authenticating Google Account...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# -------------------- QUERY BIGQUERY --------------------
from google.cloud import bigquery
import pandas as pd
from datetime import datetime
import numpy as np

print("🔄 Querying BigQuery...")

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

sql = """
WITH r AS (
  SELECT
    CASE
      WHEN a.BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
      WHEN a.BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
      WHEN a.BIT_NAME = 'BCA'                THEN 'LIVIN'
      WHEN a.BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
      WHEN a.BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
      WHEN a.BIT_NAME = 'INDODANA'           THEN 'INDODANA'
      ELSE 'SHOPEEPAY OFF US'
    END AS issuer,
    a.transaction_date,
    a.transaction_time,
    a.bank_name,
    a.comp_code,
    a.comp_name,
    a.store_code,
    a.store_name,
    a.mid,
    a.tid,
    a.yt_ref,
    a.approval_code,
    a.stan,
    a.transaction_id,
    a.bit_name,
    a.amount,
    a.mdr_amount,
    a.net_amount,
    a.upload_date,
    a.trading_date,
    a.pos_no,
    a.trx_no,
    a.promo_tag,
    a.promo_original_amount,
    DATE_ADD(a.transaction_date, INTERVAL 1 DAY) AS payment_date,
    COALESCE(
      b.ref_no,
      d.ref_no,
      LPAD(e.reff_id, 12, '0'),
      f.transaction_id,
      g.refference_number,
      RIGHT(h.merchantinvoice, 18),
      i.transaction_id,
      j.transidmerchant,
      k.reference_no,
      l.retrieval_reference_number,
      IFNULL(m.issuer_rrn, n.reference_number)
    ) AS payment_detail,
    SUM(a.amount) OVER (
      PARTITION BY
        CASE
          WHEN a.BIT_NAME = 'BNI Mobile Banking' THEN 'BNI'
          WHEN a.BIT_NAME = 'Livin by Mandiri'   THEN 'LIVIN'
          WHEN a.BIT_NAME = 'BCA'                THEN 'LIVIN'
          WHEN a.BIT_NAME = 'SHOPEEPAY'          THEN 'SHOPEEPAY'
          WHEN a.BIT_NAME = 'KREDIVO'            THEN 'KREDIVO'
          WHEN a.BIT_NAME = 'INDODANA'           THEN 'INDODANA'
          ELSE 'SHOPEEPAY OFF US'
        END
    ) AS total_payment_amount,
    a.amount AS payment_trx_recon,
    a.voucher_code,
    a.issuer_trx_id,
    COALESCE(
      b.ref_no,
      d.ref_no,
      LPAD(e.reff_id, 12, '0'),
      f.transaction_id,
      g.refference_number,
      RIGHT(h.merchantinvoice, 18),
      i.transaction_id,
      j.transidmerchant,
      k.reference_no,
      l.retrieval_reference_number,
      IFNULL(m.issuer_rrn, n.reference_number)
    ) AS key_join_issuer
  FROM `datawarehouses.yti_settlement_mcd_report_hourly` a

  -- LinkAja MSME
  LEFT JOIN (
    SELECT *
    FROM `datawarehouses.recon_linkaja_msme_report`
    WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) b
    ON b.ref_no = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(b.transaction_date)

  -- LinkAja Expansion
  LEFT JOIN (
    SELECT *
    FROM `youtap-indonesia-bi.datawarehouses.recon_linkaja_expansion_report`
    WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) d
    ON d.ref_no = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(d.transaction_date)

  -- BNI Bank
  LEFT JOIN (
    SELECT *
    FROM `datawarehouses.recon_bni_bank_report`
    WHERE DATE(trx_datetime) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) e
    ON LPAD(e.reff_id, 12, '0') = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(e.trx_datetime)

  -- ShopeePay
  LEFT JOIN (
    SELECT *
    FROM `youtap-indonesia-bi.datawarehouses.recon_shopeepay_report`
    WHERE DATE(create_time) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) f
    ON f.transaction_id = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(f.create_time)

  -- Mandiri Bank
  LEFT JOIN (
    SELECT *
    FROM `datawarehouses.recon_mandiri_bank_report`
    WHERE DATE(trxdate) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) g
    ON g.refference_number = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(g.trxdate)

  -- OVO
  LEFT JOIN (
    SELECT *
    FROM `youtap-indonesia-bi.datawarehouses.recon_ovo_report`
    WHERE DATE(transactiondate) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) h
    ON RIGHT(h.merchantinvoice, 18) = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(h.transactiondate)

  -- Kredivo
  LEFT JOIN (
    SELECT *
    FROM `datawarehouses.recon_kredivo_report`
    WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) i
    ON i.transaction_id = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(i.transaction_date)

  -- Indodana
  LEFT JOIN (
    SELECT *
    FROM `datawarehouses.recon_indodana_report`
    WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) j
    ON j.transidmerchant = a.transaction_id
    AND DATE(a.transaction_date) = DATE(j.transaction_date)

  -- BCA Bank
  LEFT JOIN (
    SELECT *
    FROM `youtap-indonesia-bi.datawarehouses.recon_issuer_bca_bank_file_report`
    WHERE DATE(payment_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) k
    ON k.reference_no = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(k.payment_date)

  -- BTN Bank
  LEFT JOIN (
    SELECT *
    FROM `youtap-indonesia-bi.datawarehouses.recon_issuer_btn_bank_file_report`
    WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) l
    ON LPAD(CAST(l.retrieval_reference_number AS STRING), 12, '0') = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(l.transaction_date)

  -- OttoPay File
  LEFT JOIN (
    SELECT *
    FROM `youtap-indonesia-bi.datawarehouses.recon_issuer_ottopay_bank_file_report`
    WHERE DATE(transaction_time) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) m
    ON m.issuer_rrn = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(m.transaction_time)

  -- OttoPay Dashboard
  LEFT JOIN (
    SELECT *
    FROM `youtap-indonesia-bi.datawarehouses.recon_issuer_ottopay_dashboard_bank_file_report`
    WHERE DATE(transaction_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
  ) n
    ON n.reference_number = a.issuer_trx_id
    AND DATE(a.transaction_date) = DATE(n.transaction_date)

  WHERE DATE(a.transaction_date) = DATE_SUB(CURRENT_DATE('Asia/Jakarta'), INTERVAL 1 DAY)
)
SELECT *
FROM r
ORDER BY transaction_date DESC, transaction_time ASC;
"""

df = client.query(sql).to_dataframe()

print(f"✅ Query complete: {len(df):,} rows, {len(df.columns)} columns")

if df.empty:
    print("⚠️ No data found. Stopping.")
    raise SystemExit

# -------------------- EXCLUDE MCD_VOUCHER FROM ALL DATA --------------------
print("\n🔧 Filtering data...")
df_original_count = len(df)
df = df[df['bit_name'] != 'MCD_VOUCHER'].copy()
mcd_voucher_excluded = df_original_count - len(df)
print(f"   ✓ Excluded {mcd_voucher_excluded:,} MCD_VOUCHER transactions")
print(f"   ✓ Remaining: {len(df):,} rows")

# -------------------- PREPARE DATA --------------------
print("\n🔢 Processing data...")

# Convert numeric columns
df['amount'] = pd.to_numeric(df['amount'], errors='coerce').fillna(0)
df['mdr_amount'] = pd.to_numeric(df['mdr_amount'], errors='coerce').fillna(0)
df['net_amount'] = pd.to_numeric(df['net_amount'], errors='coerce').fillna(0)

# Split data: MATCHED vs NOT MATCHED
df_matched = df[df['key_join_issuer'].notna() & (df['key_join_issuer'] != '')].copy()
df_not_matched = df[df['key_join_issuer'].isna() | (df['key_join_issuer'] == '')].copy()

print(f"   ✓ Matched: {len(df_matched):,} rows")
print(f"   ✓ Not Matched: {len(df_not_matched):,} rows")

# Create summary from MATCHED transactions only (filter out None/empty issuer)
df_matched_with_issuer = df_matched[df_matched['issuer'].notna() & (df_matched['issuer'] != '')].copy()
summary_data = df_matched_with_issuer.groupby('issuer').agg({
    'amount': 'sum',
    'mdr_amount': 'sum',
    'net_amount': 'sum'
}).reset_index()
summary_data.columns = ['ISSUER', 'AMOUNT', 'MDR_AMOUNT', 'NET_AMOUNT']
summary_data = summary_data.sort_values('ISSUER')

print(f"✅ Summary created from {len(df_matched_with_issuer):,} matched transactions")
print(summary_data.to_string(index=False))

# -------------------- CLEAN DATA FOR EXCEL --------------------
# Helper function to clean dataframe for Excel
def clean_for_excel(df):
    """Replace all NA/None/NaN values properly for Excel export"""
    df_clean = df.copy()

    for col in df_clean.columns:
        # Check data type
        col_dtype = str(df_clean[col].dtype)

        # Numeric columns
        if df_clean[col].dtype in ['float64', 'int64'] or 'float' in col_dtype or 'int' in col_dtype:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

        # Date/datetime columns - convert to string
        elif 'date' in col_dtype.lower() or 'time' in col_dtype.lower():
            df_clean[col] = df_clean[col].astype(str).replace(['nan', 'None', 'NaT', '<NA>', 'nat'], '')

        # All other columns - convert to string
        else:
            df_clean[col] = df_clean[col].astype(str).replace(['nan', 'None', 'NaT', '<NA>', 'nat'], '')

    return df_clean

print("\n🧹 Cleaning data for Excel export...")
df_matched_clean = clean_for_excel(df_matched)
df_not_matched_clean = clean_for_excel(df_not_matched)
print("✅ Data cleaned!")

# -------------------- GENERATE XLSX --------------------
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

print("\n📝 Generating XLSX with multiple sheets...")

wb = Workbook()
wb.remove(wb.active)

# -------------------- SHEET SUMMARY --------------------
print("  ✓ Sheet: SUMMARY")
ws_summary = wb.create_sheet('SUMMARY', 0)

# Define styles
header_fill = PatternFill(start_color='1F4E3B', end_color='1F4E3B', fill_type='solid')
header_font = Font(bold=True, color='FFFFFF', size=11)
thin_border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)

# ========== SECTION 1: TOTAL AMOUNT ==========
ws_summary['A1'] = 'TOTAL AMOUNT'
ws_summary['A1'].font = Font(bold=True, size=12)

# Add summary table headers
ws_summary['B2'] = 'ISSUER'
ws_summary['C2'] = 'AMOUNT'
ws_summary['D2'] = 'MDR_AMOUNT'
ws_summary['E2'] = 'NET_AMOUNT'

for cell in ['B2', 'C2', 'D2', 'E2']:
    ws_summary[cell].fill = header_fill
    ws_summary[cell].font = header_font
    ws_summary[cell].alignment = Alignment(horizontal='center', vertical='center')
    ws_summary[cell].border = thin_border

# Write summary data
current_row = 3
for _, row in summary_data.iterrows():
    ws_summary[f'B{current_row}'] = row['ISSUER']
    ws_summary[f'C{current_row}'] = float(row['AMOUNT'])
    ws_summary[f'D{current_row}'] = float(row['MDR_AMOUNT'])
    ws_summary[f'E{current_row}'] = float(row['NET_AMOUNT'])

    ws_summary[f'C{current_row}'].number_format = '#,##0.00'
    ws_summary[f'D{current_row}'].number_format = '#,##0.00'
    ws_summary[f'E{current_row}'].number_format = '#,##0.00'

    for col in ['B', 'C', 'D', 'E']:
        ws_summary[f'{col}{current_row}'].border = thin_border
        if col != 'B':
            ws_summary[f'{col}{current_row}'].alignment = Alignment(horizontal='right')

    current_row += 1

# Add spacing between sections
current_row += 2

# ========== SECTION 2: TRX NOT MATCH (ALL COLUMNS) ==========
ws_summary[f'A{current_row}'] = 'TRX NOT MATCH'
ws_summary[f'A{current_row}'].font = Font(bold=True, size=12, color='FF0000')
current_row += 1

# Write ALL column headers from the dataframe
if not df_not_matched_clean.empty:
    col_idx = 2  # Start from column B
    for col_name in df_not_matched_clean.columns:
        cell = ws_summary.cell(row=current_row, column=col_idx)
        cell.value = col_name
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = thin_border
        col_idx += 1

    current_row += 1

    # Write all NOT MATCHED rows with all columns
    for _, data_row in df_not_matched_clean.iterrows():
        col_idx = 2  # Start from column B
        for col_name in df_not_matched_clean.columns:
            cell = ws_summary.cell(row=current_row, column=col_idx)
            value = data_row[col_name]

            # Handle different data types
            if col_name in ['amount', 'mdr_amount', 'net_amount', 'total_payment_amount',
                           'payment_trx_recon', 'promo_original_amount']:
                try:
                    if value == '' or value == '0' or pd.isna(value):
                        cell.value = 0
                    else:
                        cell.value = float(value)
                    cell.number_format = '#,##0.00'
                    cell.alignment = Alignment(horizontal='right')
                except:
                    cell.value = 0
            else:
                cell.value = str(value) if str(value) not in ['', 'nan', 'None', 'NaT'] else ''

            cell.border = thin_border
            col_idx += 1

        current_row += 1
else:
    # If no NOT MATCHED data
    ws_summary[f'B{current_row}'] = 'No unmatched transactions'
    ws_summary[f'B{current_row}'].font = Font(italic=True, color='666666')

# Adjust column widths
ws_summary.column_dimensions['A'].width = 20
for col in ['B', 'C', 'D', 'E']:
    ws_summary.column_dimensions[col].width = 18

# Auto-adjust width for NOT MATCH columns
if not df_not_matched_clean.empty:
    for col_idx in range(2, 2 + len(df_not_matched_clean.columns)):
        col_letter = ws_summary.cell(row=1, column=col_idx).column_letter
        ws_summary.column_dimensions[col_letter].width = 15

ws_summary.freeze_panes = 'B3'

# -------------------- SHEETS PER ISSUER (MATCHED ONLY) --------------------
# Filter out None/empty issuer values and convert to list for sorting
matched_issuers = df_matched_clean[
    (df_matched_clean['issuer'] != '') &
    (df_matched_clean['issuer'] != 'nan') &
    (df_matched_clean['issuer'].notna())
]['issuer'].unique()

# Sort safely
matched_issuers_list = [str(x) for x in matched_issuers if x and str(x) not in ['', 'nan', 'None']]
matched_issuers_sorted = sorted(matched_issuers_list)

for issuer in matched_issuers_sorted:
    safe_name = str(issuer)[:31].replace('/', '-').replace('\\', '-').replace('*', '-').replace('?', '-').replace('[', '').replace(']', '').replace(':', '-')
    print(f"  ✓ Sheet: {safe_name}")

    ws = wb.create_sheet(safe_name)
    df_issuer = df_matched_clean[df_matched_clean['issuer'] == issuer]

    # Write data using cleaned dataframe
    for row in dataframe_to_rows(df_issuer, index=False, header=True):
        ws.append(row)

    # Style header row
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')

    ws.freeze_panes = 'A2'

# Save to file
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'mcd_settlement_{ts}.xlsx'
wb.save(filename)

print(f"\n✅ XLSX created: {filename}")
print(f"   Total sheets: {len(wb.sheetnames)} → {', '.join(wb.sheetnames)}")
print(f"   Matched transactions: {len(df_matched):,}")
print(f"   Unmatched transactions: {len(df_not_matched):,}")

# -------------------- SEND EMAIL VIA SMTP --------------------
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

print("\n📧 Sending email via SMTP...")

if GMAIL_APP_PASSWORD == 'PASTE_APP_PASSWORD_HERE':
    print("❌ ERROR: Please set GMAIL_APP_PASSWORD!")
    from google.colab import files
    files.download(filename)
    raise SystemExit

# -------------------- FORMAT SUMMARY TABLE HTML --------------------
def create_html_summary_table(df_summary, df_not_matched):
    """Create HTML table with matched summary and not matched preview"""
    html = """
    <style>
        table {
            border-collapse: collapse;
            font-family: Arial, sans-serif;
            font-size: 14px;
            margin: 20px 0;
            max-width: 100%;
            overflow-x: auto;
        }
        th {
            background-color: #1F4E3B;
            color: white;
            padding: 12px;
            text-align: center;
            font-weight: bold;
            border: 1px solid #ddd;
        }
        td {
            padding: 10px 12px;
            border: 1px solid #ddd;
        }
        td:first-child {
            text-align: left;
            font-weight: 500;
        }
        td:not(:first-child) {
            text-align: right;
            font-family: 'Courier New', monospace;
        }
        tr:nth-child(even) {
            background-color: #f9f9f9;
        }
        tr:hover {
            background-color: #f5f5f5;
        }
        .section-title {
            font-size: 16px;
            font-weight: bold;
            color: #1F4E3B;
            margin-top: 30px;
            margin-bottom: 10px;
        }
        .alert {
            color: #d9534f;
            font-weight: bold;
        }
    </style>

    <h3 class="section-title">📊 TOTAL AMOUNT (Matched Transactions)</h3>
    <table>
        <thead>
            <tr>
                <th>ISSUER</th>
                <th>AMOUNT</th>
                <th>MDR_AMOUNT</th>
                <th>NET_AMOUNT</th>
            </tr>
        </thead>
        <tbody>
    """

    for _, row in df_summary.iterrows():
        html += f"""
            <tr>
                <td>{row['ISSUER']}</td>
                <td>{row['AMOUNT']:,.2f}</td>
                <td>{row['MDR_AMOUNT']:,.2f}</td>
                <td>{row['NET_AMOUNT']:,.2f}</td>
            </tr>
        """

    html += """
        </tbody>
    </table>
    """

    # Add NOT MATCHED section
    if not df_not_matched.empty:
        html += f"""
        <h3 class="section-title alert">⚠️ TRX NOT MATCH ({len(df_not_matched):,} transactions)</h3>
        <p style="color: #666;">Transactions with key_join_issuer IS NULL (MCD_VOUCHER excluded) - see Excel attachment for all {len(df_not_matched.columns)} columns and {len(df_not_matched):,} rows</p>
        """

    return html

try:
    # Create email
    msg = MIMEMultipart('alternative')
    msg['From'] = FROM_EMAIL
    msg['To'] = ', '.join(TO_EMAILS)  # Multiple recipients
    msg['Subject'] = f'[BQ Export] MCD Settlement Report {ts}'

    # Create HTML table
    summary_html_table = create_html_summary_table(summary_data, df_not_matched)

    # Plain text version
    text_body = f"""Terlampir hasil export XLSX dari BigQuery.

📊 Summary per Issuer (Matched Transactions Only):

📈 Details:
- Total Rows (excl MCD_VOUCHER): {len(df):,}
- MCD_VOUCHER Excluded: {mcd_voucher_excluded:,}
- Matched: {len(df_matched):,}
- Not Matched: {len(df_not_matched):,}
- Total Sheets: {len(wb.sheetnames)}
- Sheets: {', '.join(wb.sheetnames)}
- Periode: {(datetime.now() - pd.Timedelta(days=1)).strftime('%d %b %Y')}
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}

File: {filename}

---
Auto-generated via Google Colab
"""

    # HTML version
    html_body = f"""
    <html>
    <head></head>
    <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
        <p>Terlampir hasil export XLSX dari BigQuery.</p>

        {summary_html_table}

        <h3 style="color: #1F4E3B;">📈 Details:</h3>
        <ul style="list-style-type: none; padding-left: 0;">
            <li><strong>Total Rows (excl MCD_VOUCHER):</strong> {len(df):,}</li>
            <li><strong>MCD_VOUCHER Excluded:</strong> {mcd_voucher_excluded:,}</li>
            <li><strong>Matched:</strong> {len(df_matched):,}</li>
            <li><strong>Not Matched:</strong> <span style="color: #d9534f; font-weight: bold;">{len(df_not_matched):,}</span></li>
            <li><strong>Total Sheets:</strong> {len(wb.sheetnames)}</li>
            <li><strong>Sheets:</strong> {', '.join(wb.sheetnames)}</li>
            <li><strong>Periode:</strong> {(datetime.now() - pd.Timedelta(days=1)).strftime('%d %b %Y')}</li>
            <li><strong>Generated:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S WIB')}</li>
        </ul>

        <p><strong>File:</strong> {filename}</p>

        <hr style="border: none; border-top: 1px solid #ddd; margin: 20px 0;">
        <p style="color: #666; font-size: 12px;">Auto-generated via Google Colab</p>
    </body>
    </html>
    """

    # Attach both versions
    part1 = MIMEText(text_body, 'plain')
    part2 = MIMEText(html_body, 'html')

    msg.attach(part1)
    msg.attach(part2)

    # Attach XLSX
    with open(filename, 'rb') as f:
        part = MIMEBase('application', 'vnd.openxmlformats-officedocument.spreadsheetml.sheet')
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={filename}')
    msg.attach(part)

    # Send via Gmail SMTP
    print("🔄 Connecting to Gmail SMTP server...")
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()

    print("🔑 Authenticating with App Password...")
    server.login(FROM_EMAIL, GMAIL_APP_PASSWORD)

    print("📤 Sending email...")
    server.send_message(msg)
    server.quit()

    print(f"\n✅ Email sent successfully!")
    print(f"   From: {FROM_EMAIL}")
    print(f"   To: {', '.join(TO_EMAILS)}")
    print(f"   Subject: [BQ Export] MCD Settlement Report {ts}")
    print(f"   Attachment: {filename}")
    print(f"\n📬 Check inboxes in 10-30 seconds!")

except smtplib.SMTPAuthenticationError as e:
    print(f"\n❌ Authentication failed: {str(e)}")
    print("\n🔍 Troubleshooting:")
    print("   1. Pastikan 2FA aktif di akun Gmail Anda")
    print("   2. Generate App Password baru di: https://myaccount.google.com/apppasswords")
    print("   3. Pastikan App Password 16 karakter TANPA SPASI")
    print("\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)

except Exception as e:
    print(f"\n❌ Email failed: {str(e)}")
    print(f"\n📥 Downloading file as fallback...")
    from google.colab import files
    files.download(filename)

print("\n🎉 Process complete!")
print(f"📁 File: {filename}")
print(f"📊 Matched: {len(df_matched):,} | Not Matched: {len(df_not_matched):,}")
print(f"🚫 MCD_VOUCHER Excluded: {mcd_voucher_excluded:,}")


📦 Installing dependencies...
✅ Installation complete!

🔐 Authenticating Google Account...
✅ Authentication successful!

🔄 Querying BigQuery...
✅ Query complete: 15,859 rows, 31 columns

🔧 Filtering data...
   ✓ Excluded 259 MCD_VOUCHER transactions
   ✓ Remaining: 15,600 rows

🔢 Processing data...
   ✓ Matched: 15,599 rows
   ✓ Not Matched: 1 rows
✅ Summary created from 15,599 matched transactions
          ISSUER      AMOUNT  MDR_AMOUNT   NET_AMOUNT
             BNI 136942928.0 1780258.064 1.351627e+08
        INDODANA  14366512.0  186764.656 1.417975e+07
         KREDIVO   9584513.0  124598.669 9.459914e+06
           LIVIN 623729777.0 8108487.101 6.156213e+08
       SHOPEEPAY  64516626.0  838716.138 6.367791e+07
SHOPEEPAY OFF US 298838405.0 3884899.265 2.949535e+08

🧹 Cleaning data for Excel export...
✅ Data cleaned!

📝 Generating XLSX with multiple sheets...
  ✓ Sheet: SUMMARY
  ✓ Sheet: BNI
  ✓ Sheet: INDODANA
  ✓ Sheet: KREDIVO
  ✓ Sheet: LIVIN
  ✓ Sheet: SHOPEEPAY
  ✓ Sheet: SHO